# MCP + LangGraph 动手教程

本 Notebook 演示如何用 **LangGraph ReAct Agent** 连接 **MCP Server**，加载工具并与模型对话。

与 `app.py`（Streamlit 宿主）使用同一套 `MultiServerMCPClient` 思路，适合本地实验与面试理解。

**参考**
- [MCP 介绍](https://modelcontextprotocol.io/introduction)

## 0. 环境准备

要求：**Python ≥ 3.12**，在项目根目录 `agents-master/` 下操作。

```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env   # 填入 API Key
```

> macOS 若 `pip install` 报 SSL 证书错误，可执行：
> `/Applications/Python\ 3.12/Install\ Certificates.command`

In [ ]:
import os
import sys

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI


def create_model():
    """优先使用百炼；否则使用 Anthropic。"""
    if os.getenv("DASHSCOPE_API_KEY"):
        return ChatOpenAI(
            model=os.getenv("DASHSCOPE_MODEL", "qwen-plus"),
            api_key=os.getenv("DASHSCOPE_API_KEY"),
            base_url=os.getenv(
                "DASHSCOPE_BASE_URL",
                "https://dashscope.aliyuncs.com/compatible-mode/v1",
            ),
            temperature=0,
            max_tokens=8192,
        )
    if os.getenv("ANTHROPIC_API_KEY"):
        return ChatAnthropic(
            model="claude-3-7-sonnet-latest", temperature=0, max_tokens=8192
        )
    raise ValueError("请在 .env 中配置 DASHSCOPE_API_KEY 或 ANTHROPIC_API_KEY")


model = create_model()
print("模型已就绪：", type(model).__name__)

## 1. MultiServerMCPClient（推荐）

`langchain-mcp-adapters >= 0.1` **不再支持** `async with client` 或 `await client.__aenter__()`。

正确用法：

```python
client = MultiServerMCPClient({...})
tools = await client.get_tools()   # 注意要 await
agent = create_react_agent(model, tools)
```

每次工具调用时，客户端会按需建立 MCP 会话，无需手动维持长连接。

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from utils import astream_graph

client = MultiServerMCPClient(
    {
        "time": {
            "command": sys.executable,
            "args": ["mcp_server_time.py"],
            "transport": "stdio",
        }
    }
)

tools = await client.get_tools()
print("已加载工具：", [t.name for t in tools])

agent = create_react_agent(model, tools)
await astream_graph(
    agent,
    {"messages": [HumanMessage(content="上海现在几点？")]},
)

## 2. Stdio 底层写法（了解协议即可）

Stdio 通过标准输入/输出与本地 MCP Server 通信，适合本机子进程场景（与 `config.json` 里 `"transport": "stdio"` 一致）。

一般项目直接用上一节的 `MultiServerMCPClient` 即可，本节仅帮助理解 MCP 会话建立过程。

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from utils import astream_graph

server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_server_time.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        print("工具：", [t.name for t in tools])

        agent = create_react_agent(model, tools)
        await astream_graph(
            agent,
            {"messages": [HumanMessage(content="东京现在几点？")]},
        )

## 3. RAG MCP Server

文件：`mcp_server_rag.py`（stdio 启动，**无需**事先单独跑服务进程）。

**运行前准备**
- 在 `data/` 目录放置 `sample.pdf`
- `.env` 中配置 `OPENAI_API_KEY`（用于 Embedding；若只用百炼，需自行改 embedding 实现）

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from utils import astream_graph

client = MultiServerMCPClient(
    {
        "document-retriever": {
            "command": sys.executable,
            "args": ["mcp_server_rag.py"],
            "transport": "stdio",
        }
    }
)

tools = await client.get_tools()
agent = create_react_agent(model, tools)

await astream_graph(
    agent,
    {
        "messages": [
            HumanMessage(
                content="用 retriever 工具检索：三星电子开发的生成式 AI 叫什么名字？"
            )
        ]
    },
)

## 4. 混合多个 MCP Server（Stdio）

同一 Agent 可同时挂载多个本地 MCP Server，例如「时间 + 简易 RAG」（与 `config.json` 思路一致）。

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from utils import astream_graph

client = MultiServerMCPClient(
    {
        "time": {
            "command": sys.executable,
            "args": ["mcp_server_time.py"],
            "transport": "stdio",
        },
        "document-retriever": {
            "command": sys.executable,
            "args": ["mcp_server_rag.py"],
            "transport": "stdio",
        },
    }
)

tools = await client.get_tools()
print("工具列表：", [t.name for t in tools])

prompt = (
    "你是智能助手。查时间用 get_current_time；检索文档用 retrieve。"
    "若 RAG 未配置（缺 data/sample.pdf 或 OPENAI_API_KEY），只回答时间问题。"
    "请用简体中文回答。"
)
agent = create_react_agent(
    model, tools, prompt=prompt, checkpointer=MemorySaver()
)
config = RunnableConfig(recursion_limit=30, thread_id="demo-1")

In [ ]:
await astream_graph(
    agent,
    {"messages": [HumanMessage(content="首尔现在几点？")]},
    config=config,
)

### 多轮对话（MemorySaver）

使用相同 `thread_id` 可保留短期上下文。

In [ ]:
await astream_graph(
    agent,
    {"messages": [HumanMessage(content="把上面的回答用三条要点总结")]},
    config=config,
)

## 5. LangChain 原生工具 + MCP 工具（可选）

MCP 工具可与 LangChain 内置工具混用，例如 Tavily 搜索（需 `TAVILY_API_KEY`）。

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

if not os.getenv("TAVILY_API_KEY"):
    print("跳过：未配置 TAVILY_API_KEY")
else:
    tavily = TavilySearchResults(max_results=3)
    mixed_tools = tools + [tavily]

    agent_mixed = create_react_agent(
        model,
        mixed_tools,
        prompt="你是智能助手，可使用搜索与 MCP 工具。请用简体中文回答。",
        checkpointer=MemorySaver(),
    )
    config2 = RunnableConfig(recursion_limit=30, thread_id="demo-2")

    await astream_graph(
        agent_mixed,
        {"messages": [HumanMessage(content="简要说说今天 AI 领域的一条新闻")]},
        config=config2,
    )

## 6. SSE 远程 MCP（可选）

远程 Server 通过 SSE/HTTP 暴露工具（与 `app.py` 中带 `url` 的配置一致）。

需**先在另一个终端**启动 MCP（以 `mcp_server_time.py` 为例，临时改为 SSE 启动）：

```bash
source .venv/bin/activate
cd agents-master
python -c "from mcp_server_time import mcp; mcp.run(transport='sse')"
```

默认监听 `http://localhost:8005/sse`。未启动服务时，下一格会跳过并提示。

In [ ]:
try:
    client_sse = MultiServerMCPClient(
        {
            "time": {
                "url": "http://localhost:8005/sse",
                "transport": "sse",
            }
        }
    )

    tools_sse = await client_sse.get_tools()
    print("SSE 工具：", [t.name for t in tools_sse])

    agent_sse = create_react_agent(model, tools_sse)
    await astream_graph(
        agent_sse,
        {"messages": [HumanMessage(content="北京现在几点？")]},
    )
except Exception as e:
    print("跳过 SSE 演示：请先在另一终端启动 MCP（见上一节说明）。")
    print("错误：", e)

## 7. Smithery MCP（可选，需 Node.js + API Key）

可从 [Smithery](https://smithery.ai/) 复制 JSON 配置，在 `app.py` 侧边栏或 `config.json` 中接入。

Notebook 中示例需替换 `your_smithery_api_key`，并安装 `npx`。

In [ ]:
SMITHERY_KEY = os.getenv("SMITHERY_API_KEY", "your_smithery_api_key")

if SMITHERY_KEY == "your_smithery_api_key":
    print("跳过：请设置 SMITHERY_API_KEY 或修改 SMITHERY_KEY")
else:
    client_smithery = MultiServerMCPClient(
        {
            "sequential-thinking": {
                "command": "npx",
                "args": [
                    "-y",
                    "@smithery/cli@latest",
                    "run",
                    "@smithery-ai/server-sequential-thinking",
                    "--key",
                    SMITHERY_KEY,
                ],
                "transport": "stdio",
            }
        }
    )
    smithery_tools = await client_smithery.get_tools()
    print("Smithery 工具：", [t.name for t in smithery_tools])